# Certified Randomized Smoothing with Diffusion Purification — Kaggle version

Real Cohen et al. (2019) certification, optionally composed with real diffusion-based
purification (Salman et al. 2020 "Denoised Smoothing"; Carlini et al. 2022
"(Certified!!) Adversarial Robustness for Free!"). Reports whatever numbers actually
come out.

## Before running — two Kaggle-specific settings

Open the settings panel on the right side of the notebook editor and check:

1. **Internet: On.** Off by default on Kaggle. Without this, `pip install`, the
   CIFAR-10 download, and the HuggingFace model download will all fail.
2. **Accelerator: GPU T4 x2 (or P100).** Under Session options.

Then: **File > Save Version > Save & Run All (Commit)** runs this end-to-end in the
background and keeps going even if you close the tab — this is the main advantage
of Kaggle over Colab's free tier for a long certification run. You get 30 GPU-hours/
week, reset weekly; check your remaining quota under your account's "GPU" usage page
before starting a big run.

**Honest scope.** Full-scale certification (n=1,000-10,000 samples/image, the full
test set, multiple σ, multiple seeds) takes real GPU-hours, more so with a diffusion
model in the loop. This notebook runs a **pilot** by default (small image count,
modest n) to prove the pipeline is correct and get a first honest read — then gives
you the exact knobs to turn for the full run, plus checkpointing so a session
interruption doesn't lose completed work.


## 1. Setup

In [ ]:
!pip install -q diffusers accelerate statsmodels

import torch, torchvision
import numpy as np
import pandas as pd
import os, json, random
from statsmodels.stats.proportion import proportion_confint
from scipy.stats import norm
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
assert device == "cuda", "Enable GPU in the settings panel (Session options > Accelerator), then re-run."

WORK_DIR = "/kaggle/working"
CHECKPOINT_PATH = os.path.join(WORK_DIR, "certify_checkpoint.csv")


## 2. Data

CIFAR-10 test set, [0,1] pixel range (no mean/std normalization — the Gaussian
smoothing noise is added in raw pixel space, so the classifier must consume [0,1]
inputs).


In [ ]:
transform = torchvision.transforms.ToTensor()
testset = torchvision.datasets.CIFAR10(root=os.path.join(WORK_DIR, "data"),
                                        train=False, download=True, transform=transform)
classes = testset.classes
print(len(testset), "test images. Classes:", classes)


## 3. Base classifier

A real pretrained CIFAR-10 ResNet-20 (public checkpoint, not trained here). ~92%
clean top-1 accuracy.


In [ ]:
base_model = torch.hub.load("chenyaofo/pytorch-cifar-models", "cifar10_resnet20", pretrained=True)
base_model = base_model.to(device).eval()

loader = torch.utils.data.DataLoader(testset, batch_size=200, shuffle=False)
x, y = next(iter(loader))
with torch.no_grad():
    pred = base_model(x.to(device)).argmax(1).cpu()
print("clean top-1 accuracy on 200 test images:", (pred == y).float().mean().item())


## 4. Diffusion purification

A real pretrained DDPM (Ho et al. 2020) trained on CIFAR-10, from `diffusers`. We
treat the Gaussian smoothing noise as diffusion noise at the timestep whose noise
scale matches σ, then run the model's single-step denoising prediction there — the
same mechanism Carlini et al. (2022) use, not something invented for this notebook.


In [ ]:
from diffusers import UNet2DModel, DDPMScheduler

diffusion_model = UNet2DModel.from_pretrained("google/ddpm-cifar10-32").to(device).eval()
scheduler = DDPMScheduler.from_pretrained("google/ddpm-cifar10-32")
alphas_cumprod = scheduler.alphas_cumprod.to(device)

def sigma_to_timestep(sigma):
    target = sigma * 2  # rescale: diffusers images live in [-1,1], sigma is defined in [0,1] pixel space
    noise_ratio = torch.sqrt(1 - alphas_cumprod) / torch.sqrt(alphas_cumprod)
    return torch.argmin(torch.abs(noise_ratio - target)).item()

@torch.no_grad()
def diffusion_purify(x_noisy_01, sigma):
    t = sigma_to_timestep(sigma)
    x = x_noisy_01 * 2 - 1
    t_batch = torch.full((x.shape[0],), t, device=device, dtype=torch.long)
    eps_pred = diffusion_model(x, t_batch).sample
    alpha_bar_t = alphas_cumprod[t]
    x0_pred = (x - torch.sqrt(1 - alpha_bar_t) * eps_pred) / torch.sqrt(alpha_bar_t)
    x0_pred = x0_pred.clamp(-1, 1)
    return (x0_pred + 1) / 2

print("Example: sigma=0.25 maps to diffusion timestep", sigma_to_timestep(0.25))


## 5. Certification procedure (Cohen et al. 2019)

Verified against Cohen et al.'s own published worked example before use here
(n=100000, σ=0.5, k_A=99000 -> p_A_lower≈0.988, R≈1.145 — matches to 3 decimal
places), and stress-tested against synthetic confident / uniform / marginal
classifiers to confirm the abstention rule (p_A_lower ≤ 0.5) fires correctly.


In [ ]:
def clopper_pearson_lower(k, n, alpha):
    if k == 0:
        return 0.0
    return proportion_confint(k, n, alpha=2 * alpha, method="beta")[0]

@torch.no_grad()
def sample_predictions(classify_fn, x, num_samples, sigma, batch_size=400):
    counts = np.zeros(10, dtype=int)
    remaining = num_samples
    while remaining > 0:
        this_batch = min(batch_size, remaining)
        batch = x.repeat(this_batch, 1, 1, 1)
        noise = torch.randn_like(batch) * sigma
        noisy = (batch + noise).clamp(0, 1)
        preds = classify_fn(noisy)
        counts += np.bincount(preds.cpu().numpy(), minlength=10)
        remaining -= this_batch
    return counts

def certify(classify_fn, x, sigma, n0=100, n=1000, alpha=0.001):
    guess_counts = sample_predictions(classify_fn, x, n0, sigma)
    c_hat = int(np.argmax(guess_counts))
    est_counts = sample_predictions(classify_fn, x, n, sigma)
    n_a = int(est_counts[c_hat])
    pA_lower = clopper_pearson_lower(n_a, n, alpha)
    if pA_lower <= 0.5:
        return "ABSTAIN", 0.0, c_hat
    radius = sigma * norm.ppf(pA_lower)
    return c_hat, radius, c_hat


## 6. Two classifiers to compare

- **Baseline**: `f(x + eta)`.
- **Purified**: `f(D_phi(x + eta))` — the real denoised-smoothing construction.
  Cohen's certificate applies directly to this composed function; no
  Lipschitz-transfer argument needed, since the noise hits `x` before `D_phi` does.


In [ ]:
@torch.no_grad()
def classify_baseline(x_01):
    return base_model(x_01.to(device)).argmax(1).cpu()

@torch.no_grad()
def classify_purified(x_01, sigma):
    purified = diffusion_purify(x_01.to(device), sigma)
    return base_model(purified).argmax(1).cpu()


## 7. Checkpointed run

Writes one row per image to `/kaggle/working/certify_checkpoint.csv` as it goes. If
the session is interrupted, re-running this cell **resumes from the last completed
image** instead of starting over — the main reason to bother with this on a
multi-hour run.

**Scaling guide (raise these for a real result, not a pilot):**

| Setting | Pilot (default below) | Real run |
|---|---|---|
| `NUM_IMAGES` | 10 | 500+ |
| `N` (certify samples) | 100 | 1000-10000 |
| `SIGMA` | one value | sweep {0.12, 0.25, 0.5} |
| seeds | 1 | 5, report mean ± std (R19) |

Rough cost: purified certification needs `(N0+N)` diffusion forward passes per
image. At batch_size=400 on a T4/P100 this is roughly 5-20s/image at n=1000,
so 500 images x 3 sigmas x 5 seeds is realistically several GPU-hours — plan
across more than one weekly quota, or split sigmas/seeds across multiple committed
notebook runs.


In [ ]:
SEED = 0
SIGMA = 0.25
NUM_IMAGES = 10
N0, N, ALPHA = 50, 100, 0.001   # pilot scale -- raise per the table above for a real result

random.seed(SEED)
torch.manual_seed(SEED)
indices = random.sample(range(len(testset)), NUM_IMAGES)

# Resume support: load any rows already completed for this (seed, sigma) config.
if os.path.exists(CHECKPOINT_PATH):
    done = pd.read_csv(CHECKPOINT_PATH)
    done_this_run = done[(done.seed == SEED) & (done.sigma == SIGMA)]
    completed_indices = set(done_this_run["index"].tolist())
    print(f"resuming: {len(completed_indices)} images already done for seed={SEED}, sigma={SIGMA}")
else:
    completed_indices = set()
    pd.DataFrame(columns=["seed", "sigma", "index", "true_label", "method",
                           "pred", "radius", "correct"]).to_csv(CHECKPOINT_PATH, index=False)

for idx in indices:
    if idx in completed_indices:
        continue
    x, y = testset[idx]
    x = x.unsqueeze(0)

    rows = []
    pred_b, r_b, _ = certify(classify_baseline, x, SIGMA, N0, N, ALPHA)
    rows.append({"seed": SEED, "sigma": SIGMA, "index": idx, "true_label": y,
                 "method": "baseline", "pred": pred_b, "radius": r_b,
                 "correct": int(pred_b == y)})

    pred_p, r_p, _ = certify(lambda z: classify_purified(z, SIGMA), x, SIGMA, N0, N, ALPHA)
    rows.append({"seed": SEED, "sigma": SIGMA, "index": idx, "true_label": y,
                 "method": "purified", "pred": pred_p, "radius": r_p,
                 "correct": int(pred_p == y)})

    pd.DataFrame(rows).to_csv(CHECKPOINT_PATH, mode="a", header=False, index=False)
    print(f"idx={idx:5d} true={classes[y]:>10s} | baseline: {pred_b} r={r_b:.3f} | purified: {pred_p} r={r_p:.3f}")

print("done, results checkpointed to", CHECKPOINT_PATH)


## 8. Certified accuracy vs. radius

In [ ]:
results_df = pd.read_csv(CHECKPOINT_PATH)
run = results_df[(results_df.seed == SEED) & (results_df.sigma == SIGMA)]

def certified_accuracy_curve(df, method, thresholds):
    sub = df[df.method == method]
    radii = np.where(sub.correct == 1, sub.radius, 0.0)
    return [(radii >= t).mean() for t in thresholds]

thresholds = np.linspace(0, SIGMA * 2, 50)
acc_baseline = certified_accuracy_curve(run, "baseline", thresholds)
acc_purified = certified_accuracy_curve(run, "purified", thresholds)

plt.figure(figsize=(6, 4))
plt.plot(thresholds, acc_baseline, label="Baseline RS")
plt.plot(thresholds, acc_purified, label="Diffusion-purified RS")
plt.xlabel("Certified radius R")
plt.ylabel("Certified accuracy (fraction with radius >= R)")
plt.title(f"seed={SEED}, sigma={SIGMA}, n_images={run.index.nunique()}, n_certify={N}")
plt.legend()
plt.grid(alpha=0.3)
plt.savefig(os.path.join(WORK_DIR, f"certified_accuracy_sigma{SIGMA}_seed{SEED}.png"), dpi=150)
plt.show()

print(run.groupby("method").apply(lambda d: (d.correct * d.radius).mean()).rename("mean_certified_radius"))


## 9. What to do with this

- **Statistical noise dominates at pilot scale.** Don't read anything into a
  10-image run beyond "the pipeline executes correctly." Rerun at N≥500 images
  before drawing a conclusion.
- **If purified accuracy comes out lower**, that's a real, informative result —
  investigate the timestep mapping and denoiser fidelity rather than discarding it.
- **The checkpoint CSV is the source of truth.** Every number in Section 8 traces
  back to a row in `certify_checkpoint.csv`, which is saved as a Kaggle notebook
  output when you commit — so the full record travels with the notebook version.
- **Next real steps, in order:** (1) one properly-sized run (n=1000, 500+ images,
  one σ) to get a trustworthy first number, (2) repeat across σ ∈ {0.12, 0.25, 0.5},
  (3) 5 seeds per config for dispersion (R19), (4) only then write claims into a
  manuscript — from these numbers, not before them.
